In [0]:
from pyspark.sql.functions import avg, abs, col, when


In [0]:
silver_sales_path = "/Volumes/workspace/m5_project/silver/silver_sales"

silver_sales_df = spark.read.parquet(
    silver_sales_path
)

display(silver_sales_df.limit(10))

id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk
HOUSEHOLD_1_363_TX_3_validation,HOUSEHOLD_1_363,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_364_TX_3_validation,HOUSEHOLD_1_364,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_365_TX_3_validation,HOUSEHOLD_1_365,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_366_TX_3_validation,HOUSEHOLD_1_366,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_367_TX_3_validation,HOUSEHOLD_1_367,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_368_TX_3_validation,HOUSEHOLD_1_368,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_369_TX_3_validation,HOUSEHOLD_1_369,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_370_TX_3_validation,HOUSEHOLD_1_370,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,2,2011-01-29,11101
HOUSEHOLD_1_371_TX_3_validation,HOUSEHOLD_1_371,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_372_TX_3_validation,HOUSEHOLD_1_372,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,6,2011-01-29,11101


In [0]:
baseline_df = silver_sales_df.groupBy(
    "item_id",
    "store_id",
    "cat_id",
    "dept_id",
    "state_id"
).agg(
    avg("sales").alias("expected_sales")
)

display(baseline_df.limit(10))


item_id,store_id,cat_id,dept_id,state_id,expected_sales
HOUSEHOLD_2_006,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.07631991636173549
HOUSEHOLD_2_050,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.14584422373235756
HOUSEHOLD_2_056,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.09984317825405123
HOUSEHOLD_2_122,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.20543648719289076
HOUSEHOLD_2_127,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.11029796131730267
HOUSEHOLD_2_232,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.29796131730266595
HOUSEHOLD_2_285,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.14636696288552012
HOUSEHOLD_2_462,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.11395713538944068
HOUSEHOLD_2_468,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,0.2624150548876111
FOODS_1_107,TX_3,FOODS,FOODS_1,TX,1.0862519602718244


In [0]:
# anomaly_df = silver_sales_df.join(
#     baseline_df,
#     on=["item_id", "store_id"],
#     how="left"
# )

# display(anomaly_df.limit(10))

anomaly_df = silver_sales_df.join(
    baseline_df,
    on=[
        "item_id",
        "store_id",
        "cat_id",
        "dept_id",
        "state_id"
    ],
    how="left"
)

display(anomaly_df.limit(10))


item_id,store_id,cat_id,dept_id,state_id,id,day,sales,date,wm_yr_wk,expected_sales
HOUSEHOLD_1_363,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_363_TX_3_validation,d_1,0,2011-01-29,11101,0.31364349189754315
HOUSEHOLD_1_364,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_364_TX_3_validation,d_1,0,2011-01-29,11101,0.41400940930475694
HOUSEHOLD_1_365,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_365_TX_3_validation,d_1,0,2011-01-29,11101,0.6278097229482488
HOUSEHOLD_1_366,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_366_TX_3_validation,d_1,0,2011-01-29,11101,0.872974385781495
HOUSEHOLD_1_367,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_367_TX_3_validation,d_1,0,2011-01-29,11101,0.1991636173549399
HOUSEHOLD_1_368,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_368_TX_3_validation,d_1,0,2011-01-29,11101,0.1693674856246733
HOUSEHOLD_1_369,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_369_TX_3_validation,d_1,0,2011-01-29,11101,1.002613695765813
HOUSEHOLD_1_370,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_370_TX_3_validation,d_1,2,2011-01-29,11101,0.7971772085729221
HOUSEHOLD_1_371,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_371_TX_3_validation,d_1,0,2011-01-29,11101,0.3303711447987454
HOUSEHOLD_1_372,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_372_TX_3_validation,d_1,6,2011-01-29,11101,0.9560899111343439


calculate the anamoly score

In [0]:
anomaly_df = (
    anomaly_df
    .withColumn("category", col("cat_id"))
    .withColumn("department", col("dept_id"))
)

In [0]:
anomaly_df = anomaly_df.withColumn(
    "anomaly_score",
    abs(
        col("sales") - col("expected_sales")
    )
)

Flag major anamolies

In [0]:
# anomaly_df = anomaly_df.withColumn(
#     "anomaly_flag",
#     when(
#         col("anomaly_score") > 5,
#         "HIGH"
#     ).when(
#         col("anomaly_score") > 2,
#         "MEDIUM"
#     ).otherwise(
#         "LOW"
#     )
# )

high_anomaly_threshold = anomaly_df.approxQuantile(
    "anomaly_score",
    [0.90],
    0.01
)[0]

medium_anomaly_threshold = anomaly_df.approxQuantile(
    "anomaly_score",
    [0.75],
    0.01
)[0]

print("High anomaly threshold:", high_anomaly_threshold)
print("Medium anomaly threshold:", medium_anomaly_threshold)


anomaly_df = anomaly_df.withColumn(
    "anomaly_flag",

    when(
        col("anomaly_score") > high_anomaly_threshold,
        "HIGH"
    )

    .when(
        col("anomaly_score") > medium_anomaly_threshold,
        "MEDIUM"
    )

    .otherwise(
        "LOW"
    )
)

display(
    anomaly_df.select(
        "item_id",
        "category",
        "department",
        "store_id",
        "state_id",
        "date",
        "sales",
        "expected_sales",
        "anomaly_score",
        "anomaly_flag"
    ).limit(20)
)


High anomaly threshold: 2.38212232096184
Medium anomaly threshold: 1.033978044955567


item_id,category,department,store_id,state_id,date,sales,expected_sales,anomaly_score,anomaly_flag
HOUSEHOLD_1_371,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,0.3303711447987454,0.3303711447987454,LOW
HOUSEHOLD_1_367,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,0.1991636173549399,0.1991636173549399,LOW
HOUSEHOLD_1_379,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,4,4.2728698379508625,0.2728698379508625,LOW
HOUSEHOLD_1_375,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,0.1657083115525353,0.1657083115525353,LOW
HOUSEHOLD_1_366,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,0.872974385781495,0.872974385781495,LOW
HOUSEHOLD_1_380,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,0.35964453737584945,0.35964453737584945,LOW
HOUSEHOLD_1_378,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,0.037114479874542604,0.037114479874542604,LOW
HOUSEHOLD_1_369,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,1.002613695765813,1.002613695765813,LOW
HOUSEHOLD_1_376,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,0,0.46994249869315213,0.46994249869315213,LOW
HOUSEHOLD_1_370,HOUSEHOLD,HOUSEHOLD_1,TX_3,TX,2011-01-29,2,0.7971772085729221,1.202822791427078,MEDIUM


In [0]:
gold_anomaly_path = "/Volumes/workspace/m5_project/gold/gold_demand_anomaly"

anomaly_df.write.mode("overwrite").parquet(
    gold_anomaly_path
)

print("Gold Demand Anomaly written successfully")

Gold Demand Anomaly written successfully


In [0]:
gold_anomaly_check = spark.read.parquet(
    gold_anomaly_path
)

display(gold_anomaly_check.limit(5))

item_id,store_id,cat_id,dept_id,state_id,id,day,sales,date,wm_yr_wk,expected_sales,category,department,anomaly_score,anomaly_flag
HOUSEHOLD_1_363,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_363_TX_3_validation,d_1,0,2011-01-29,11101,0.31364349189754315,HOUSEHOLD,HOUSEHOLD_1,0.31364349189754315,LOW
HOUSEHOLD_1_364,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_364_TX_3_validation,d_1,0,2011-01-29,11101,0.41400940930475694,HOUSEHOLD,HOUSEHOLD_1,0.41400940930475694,LOW
HOUSEHOLD_1_365,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_365_TX_3_validation,d_1,0,2011-01-29,11101,0.6278097229482488,HOUSEHOLD,HOUSEHOLD_1,0.6278097229482488,LOW
HOUSEHOLD_1_366,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_366_TX_3_validation,d_1,0,2011-01-29,11101,0.872974385781495,HOUSEHOLD,HOUSEHOLD_1,0.872974385781495,LOW
HOUSEHOLD_1_367,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,HOUSEHOLD_1_367_TX_3_validation,d_1,0,2011-01-29,11101,0.1991636173549399,HOUSEHOLD,HOUSEHOLD_1,0.1991636173549399,LOW


In [0]:
gold_anomaly_export_path = "/Volumes/workspace/m5_project/gold/exports/gold_demand_anomaly_csv"

anomaly_df.write.mode("overwrite") \
    .option("header", True) \
    .csv(gold_anomaly_export_path)

print("Demand Anomaly CSV export complete")

Demand Anomaly CSV export complete
